# Tugas Pemrograman 2 - Search Engine "from Scratch"

## Temu-Balik Informasi

Notebook ini berisi implementasi search engine dengan fitur-fitur:
1. **Kompresi Index**: Standard, VBE (Variable Byte Encoding), dan OptPForDelta (bit-level)
2. **Scoring**: TF-IDF dan BM25
3. **Optimasi**: WAND Top-K Retrieval
4. **Evaluasi**: RBP, DCG, NDCG, dan Average Precision (AP)

---

## 1. Setup dan Import

In [1]:
import os
import random

from bsbi import BSBIIndex
from compression import StandardPostings, VBEPostings, OptPForDeltaPostings
from evaluation import rbp, dcg, ndcg, ap, load_qrels, eval_all

print("Semua modul berhasil diimport!")

Semua modul berhasil diimport!


---
## 2. Test Algoritma Kompresi

Perbandingan tiga algoritma kompresi:
- **StandardPostings**: Tanpa kompresi (4 bytes per integer)
- **VBEPostings**: Variable Byte Encoding
- **OptPForDelta**: Optimized Patched Frame-of-Reference Delta (bit-level)

In [2]:
# Test dengan data kecil
postings_list = [34, 67, 89, 454, 2345738]
tf_list = [12, 10, 3, 4, 1]

print("=" * 60)
print("TEST KOMPRESI - DATA KECIL")
print("=" * 60)
print(f"Postings list: {postings_list}")
print(f"TF list      : {tf_list}")
print()

for Postings in [StandardPostings, VBEPostings, OptPForDeltaPostings]:
    print(f"--- {Postings.__name__} ---")
    
    # Encode
    encoded_postings = Postings.encode(postings_list)
    encoded_tf = Postings.encode_tf(tf_list)
    
    # Decode
    decoded_postings = Postings.decode(encoded_postings)
    decoded_tf = Postings.decode_tf(encoded_tf)
    
    # Verify
    assert decoded_postings == postings_list, "Postings decode error!"
    assert decoded_tf == tf_list, "TF decode error!"
    
    print(f"Ukuran postings: {len(encoded_postings)} bytes")
    print(f"Ukuran TF      : {len(encoded_tf)} bytes")
    print(f"Decode OK      : Ya")
    print()

TEST KOMPRESI - DATA KECIL
Postings list: [34, 67, 89, 454, 2345738]
TF list      : [12, 10, 3, 4, 1]

--- StandardPostings ---
Ukuran postings: 20 bytes
Ukuran TF      : 20 bytes
Decode OK      : Ya

--- VBEPostings ---
Ukuran postings: 9 bytes
Ukuran TF      : 5 bytes
Decode OK      : Ya

--- OptPForDeltaPostings ---
Ukuran postings: 24 bytes
Ukuran TF      : 13 bytes
Decode OK      : Ya



In [3]:
# Test dengan data besar (500 dokumen)
print("=" * 60)
print("TEST KOMPRESI - DATA BESAR (500 postings)")
print("=" * 60)

random.seed(42)  # Untuk reproducibility
large_postings = sorted(random.sample(range(1, 100000), 500))
large_tf = [random.randint(1, 100) for _ in range(500)]

results = []
for Postings in [StandardPostings, VBEPostings, OptPForDeltaPostings]:
    encoded_p = Postings.encode(large_postings)
    encoded_tf = Postings.encode_tf(large_tf)
    decoded_p = Postings.decode(encoded_p)
    decoded_tf = Postings.decode_tf(encoded_tf)
    
    assert decoded_p == large_postings, f"{Postings.__name__} postings decode failed"
    assert decoded_tf == large_tf, f"{Postings.__name__} TF decode failed"
    
    results.append({
        'name': Postings.__name__,
        'postings_size': len(encoded_p),
        'tf_size': len(encoded_tf)
    })

# Print comparison table
print(f"{'Algorithm':<25} {'Postings (bytes)':<18} {'TF (bytes)':<15} {'Total':<12}")
print("-" * 70)
for r in results:
    total = r['postings_size'] + r['tf_size']
    print(f"{r['name']:<25} {r['postings_size']:<18} {r['tf_size']:<15} {total:<12}")

# Calculate compression ratio
baseline = results[0]['postings_size'] + results[0]['tf_size']
print()
print("Compression Ratio (vs Standard):")
for r in results:
    total = r['postings_size'] + r['tf_size']
    ratio = baseline / total
    savings = (1 - total/baseline) * 100
    print(f"  {r['name']}: {ratio:.2f}x ({savings:.1f}% smaller)")

TEST KOMPRESI - DATA BESAR (500 postings)
Algorithm                 Postings (bytes)   TF (bytes)      Total       
----------------------------------------------------------------------
StandardPostings          2000               2000            4000        
VBEPostings               770                500             1270        
OptPForDeltaPostings      668                466             1134        

Compression Ratio (vs Standard):
  StandardPostings: 1.00x (0.0% smaller)
  VBEPostings: 3.15x (68.2% smaller)
  OptPForDeltaPostings: 3.53x (71.7% smaller)


---
## 3. Indexing dengan BSBI

Membuat inverted index dari koleksi dokumen menggunakan skema BSBI (Blocked Sort-Based Indexing).

**Catatan:** Jika index sudah ada, proses indexing akan di-skip.

In [4]:
# Cek apakah index sudah ada
index_exists = os.path.exists('index/main_index.index') and os.path.exists('index/main_index.dict')

if index_exists:
    print("Index sudah ada!")
    print("  - main_index.index")
    print("  - main_index.dict")
    print("  - terms.dict")
    print("  - docs.dict")
    print("\nUntuk rebuild, hapus folder index/ dan jalankan ulang cell ini.")
else:
    print("Index belum ada. Memulai proses indexing...")
    print("(Proses ini membutuhkan waktu beberapa menit)\n")
    
    BSBI_instance = BSBIIndex(
        data_dir='collection',
        postings_encoding=VBEPostings,
        output_dir='index'
    )
    BSBI_instance.index()
    
    print("\nIndexing selesai!")

Index sudah ada!
  - main_index.index
  - main_index.dict
  - terms.dict
  - docs.dict

Untuk rebuild, hapus folder index/ dan jalankan ulang cell ini.


---
## 4. Search Engine Demo

Mendemonstrasikan tiga metode retrieval:
1. **TF-IDF**: Term Frequency - Inverse Document Frequency
2. **BM25**: Best Matching 25 (Okapi BM25)
3. **BM25 + WAND**: BM25 dengan optimasi WAND Top-K

In [5]:
# Inisialisasi Search Engine
BSBI_instance = BSBIIndex(
    data_dir='collection',
    postings_encoding=VBEPostings,
    output_dir='index'
)

# Load index
BSBI_instance.load()
print(f"Index loaded!")
print(f"  Total terms: {len(BSBI_instance.term_id_map)}")
print(f"  Total docs : {len(BSBI_instance.doc_id_map)}")

Index loaded!
  Total terms: 20219
  Total docs : 1033


In [6]:
# Queries untuk demo
queries = [
    "alkylated with radioactive iodoacetate",
    "psychodrama for disturbed children",
    "lipid metabolism in toxemia and normal pregnancy"
]

### 4.1 TF-IDF Retrieval

In [7]:
# Demo TF-IDF Retrieval
print("=" * 70)
print("TF-IDF RETRIEVAL")
print("=" * 70)

for query in queries:
    print(f"\nQuery: {query}")
    print("-" * 50)
    results = BSBI_instance.retrieve_tfidf(query, k=5)
    for i, (score, doc) in enumerate(results, 1):
        doc_name = os.path.basename(doc)
        print(f"  {i}. {doc_name:<15} (score: {score:.4f})")

TF-IDF RETRIEVAL

Query: alkylated with radioactive iodoacetate
--------------------------------------------------
  1. 507.txt         (score: 28.2394)
  2. 388.txt         (score: 9.7620)
  3. 247.txt         (score: 7.9293)
  4. 745.txt         (score: 7.9293)
  5. 512.txt         (score: 7.5907)

Query: psychodrama for disturbed children
--------------------------------------------------
  1. 820.txt         (score: 21.2664)
  2. 918.txt         (score: 14.0125)
  3. 821.txt         (score: 13.9154)
  4. 799.txt         (score: 11.9780)
  5. 926.txt         (score: 11.3003)

Query: lipid metabolism in toxemia and normal pregnancy
--------------------------------------------------
  1. 7.txt           (score: 31.2397)
  2. 159.txt         (score: 18.8491)
  3. 81.txt          (score: 12.4148)
  4. 12.txt          (score: 11.4552)
  5. 329.txt         (score: 11.3178)


### 4.2 BM25 Retrieval

BM25 menggunakan formula:
```
score(Q, D) = sum IDF(t) x [TF(t,D) x (k1+1)] / [TF(t,D) + k1 x (1-b + b x |D|/avgdl)]
```
- **k1**: parameter saturasi TF (default: 1.5)
- **b**: parameter normalisasi panjang dokumen (default: 0.75)

In [8]:
# Demo BM25 Retrieval
print("=" * 70)
print("BM25 RETRIEVAL (k1=1.5, b=0.75)")
print("=" * 70)

for query in queries:
    print(f"\nQuery: {query}")
    print("-" * 50)
    results = BSBI_instance.retrieve_bm25(query, k=5, k1=1.5, b=0.75)
    for i, (score, doc) in enumerate(results, 1):
        doc_name = os.path.basename(doc)
        print(f"  {i}. {doc_name:<15} (score: {score:.4f})")

BM25 RETRIEVAL (k1=1.5, b=0.75)

Query: alkylated with radioactive iodoacetate
--------------------------------------------------
  1. 507.txt         (score: 23.8817)
  2. 793.txt         (score: 7.7019)
  3. 247.txt         (score: 7.0714)
  4. 745.txt         (score: 6.8166)
  5. 388.txt         (score: 6.0272)

Query: psychodrama for disturbed children
--------------------------------------------------
  1. 820.txt         (score: 20.3185)
  2. 821.txt         (score: 11.5228)
  3. 918.txt         (score: 10.4808)
  4. 36.txt          (score: 9.5742)
  5. 926.txt         (score: 8.9988)

Query: lipid metabolism in toxemia and normal pregnancy
--------------------------------------------------
  1. 7.txt           (score: 27.5577)
  2. 159.txt         (score: 13.5196)
  3. 371.txt         (score: 9.7252)
  4. 328.txt         (score: 9.4435)
  5. 12.txt          (score: 8.7496)


### 4.3 BM25 + WAND Top-K Retrieval

**WAND (Weak AND)** adalah algoritma optimasi yang menghindari scoring semua dokumen dengan menggunakan upper bound pada kontribusi setiap term. Dokumen yang tidak mungkin masuk top-K akan di-skip.

In [9]:
# Demo BM25 + WAND Retrieval
print("=" * 70)
print("BM25 + WAND RETRIEVAL (k1=1.5, b=0.75)")
print("=" * 70)

for query in queries:
    print(f"\nQuery: {query}")
    print("-" * 50)
    results = BSBI_instance.retrieve_bm25_wand(query, k=5, k1=1.5, b=0.75)
    for i, (score, doc) in enumerate(results, 1):
        doc_name = os.path.basename(doc)
        print(f"  {i}. {doc_name:<15} (score: {score:.4f})")

BM25 + WAND RETRIEVAL (k1=1.5, b=0.75)

Query: alkylated with radioactive iodoacetate
--------------------------------------------------
  1. 507.txt         (score: 23.8817)
  2. 793.txt         (score: 7.7019)
  3. 247.txt         (score: 7.0714)
  4. 745.txt         (score: 6.8166)
  5. 388.txt         (score: 6.0272)

Query: psychodrama for disturbed children
--------------------------------------------------
  1. 820.txt         (score: 20.3185)
  2. 821.txt         (score: 11.5228)
  3. 918.txt         (score: 10.4808)
  4. 36.txt          (score: 9.5742)
  5. 926.txt         (score: 8.9988)

Query: lipid metabolism in toxemia and normal pregnancy
--------------------------------------------------
  1. 7.txt           (score: 27.5577)
  2. 159.txt         (score: 13.5196)
  3. 371.txt         (score: 9.7252)
  4. 328.txt         (score: 9.4435)
  5. 12.txt          (score: 8.7496)


### 4.4 Perbandingan Metode Retrieval

In [10]:
# Perbandingan hasil dari ketiga metode untuk satu query
query = "alkylated with radioactive iodoacetate"
k = 10

print("=" * 70)
print("PERBANDINGAN METODE RETRIEVAL")
print(f"Query: {query}")
print("=" * 70)

tfidf_results = BSBI_instance.retrieve_tfidf(query, k=k)
bm25_results = BSBI_instance.retrieve_bm25(query, k=k)
wand_results = BSBI_instance.retrieve_bm25_wand(query, k=k)

print(f"\n{'Rank':<6} {'TF-IDF':<20} {'BM25':<20} {'BM25+WAND':<20}")
print("-" * 66)

for i in range(k):
    tfidf_doc = os.path.basename(tfidf_results[i][1]) if i < len(tfidf_results) else "-"
    bm25_doc = os.path.basename(bm25_results[i][1]) if i < len(bm25_results) else "-"
    wand_doc = os.path.basename(wand_results[i][1]) if i < len(wand_results) else "-"
    print(f"{i+1:<6} {tfidf_doc:<20} {bm25_doc:<20} {wand_doc:<20}")

# Verifikasi BM25 dan WAND menghasilkan hasil yang sama
bm25_docs = [doc for _, doc in bm25_results]
wand_docs = [doc for _, doc in wand_results]

print()
if bm25_docs == wand_docs:
    print("[OK] BM25 dan BM25+WAND menghasilkan ranking yang sama!")
else:
    print("[INFO] BM25 dan BM25+WAND menghasilkan ranking berbeda (normal untuk tie-breaking)")

PERBANDINGAN METODE RETRIEVAL
Query: alkylated with radioactive iodoacetate

Rank   TF-IDF               BM25                 BM25+WAND           
------------------------------------------------------------------
1      507.txt              507.txt              507.txt             
2      388.txt              793.txt              793.txt             
3      247.txt              247.txt              247.txt             
4      745.txt              745.txt              745.txt             
5      512.txt              388.txt              388.txt             
6      793.txt              512.txt              512.txt             
7      393.txt              119.txt              119.txt             
8      557.txt              381.txt              381.txt             
9      1001.txt             261.txt              261.txt             
10     119.txt              1018.txt             1018.txt            

[OK] BM25 dan BM25+WAND menghasilkan ranking yang sama!


---
## 5. Evaluasi Metrik IR

Menghitung kualitas search engine dengan 4 metrik:
1. **RBP (Rank-Biased Precision)**: Mempertimbangkan probabilitas user berhenti membaca
2. **DCG (Discounted Cumulative Gain)**: Memberikan bobot lebih tinggi pada dokumen relevan di posisi atas
3. **NDCG (Normalized DCG)**: DCG dinormalisasi dengan IDCG (ideal DCG)
4. **AP (Average Precision)**: Rata-rata precision pada setiap posisi dokumen relevan

### 5.1 Test Metrik dengan Contoh Sederhana

In [11]:
# Test metrik dengan contoh sederhana
print("=" * 50)
print("TEST METRIK EVALUASI")
print("=" * 50)

# Contoh ranking: dokumen di posisi 1,3,4,5,8 relevan
test_ranking = [1, 0, 1, 1, 1, 0, 0, 1, 0, 0]
print(f"\nTest ranking: {test_ranking}")
print(f"Jumlah relevan: {sum(test_ranking)}")
print(f"RBP score   : {rbp(test_ranking):.4f}")
print(f"DCG@10 score: {dcg(test_ranking, k=10):.4f}")
print(f"NDCG@10 score: {ndcg(test_ranking, k=10):.4f}")
print(f"AP score    : {ap(test_ranking):.4f}")

# Test dengan perfect ranking (semua relevan di atas)
perfect_ranking = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]
print(f"\nPerfect ranking: {perfect_ranking}")
print(f"NDCG@10 score: {ndcg(perfect_ranking, k=10):.4f}")  # Harus = 1.0
print(f"AP score    : {ap(perfect_ranking):.4f}")  # Harus = 1.0

# Test dengan worst ranking (semua relevan di bawah)
worst_ranking = [0, 0, 0, 0, 0, 1, 1, 1, 1, 1]
print(f"\nWorst ranking: {worst_ranking}")
print(f"NDCG@10 score: {ndcg(worst_ranking, k=10):.4f}")
print(f"AP score    : {ap(worst_ranking):.4f}")

TEST METRIK EVALUASI

Test ranking: [1, 0, 1, 1, 1, 0, 0, 1, 0, 0]
Jumlah relevan: 5
RBP score   : 0.5543
DCG@10 score: 2.6330
NDCG@10 score: 0.8930
AP score    : 0.7683

Perfect ranking: [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]
NDCG@10 score: 1.0000
AP score    : 1.0000

Worst ranking: [0, 0, 0, 0, 0, 1, 1, 1, 1, 1]
NDCG@10 score: 0.5410
AP score    : 0.3544


### 5.2 Evaluasi pada Dataset (30 Queries)

In [12]:
# Load query relevance judgments
qrels = load_qrels()

# Verifikasi qrels loaded correctly
assert qrels["Q1"][166] == 1, "qrels error"
assert qrels["Q1"][300] == 0, "qrels error"
print("Query relevance judgments loaded!")
print(f"Total queries: 30")

Query relevance judgments loaded!
Total queries: 30


In [13]:
# Evaluasi TF-IDF vs BM25
eval_all(qrels)

EVALUASI TF-IDF vs BM25

--- TF-IDF ---
Hasil evaluasi TFIDF terhadap 30 queries
RBP score     = 0.5980
DCG@10 score  = 2.7983
NDCG@10 score = 0.6431
AP score      = 0.4948

--- BM25 ---
Hasil evaluasi BM25 terhadap 30 queries
RBP score     = 0.6353
DCG@10 score  = 2.9911
NDCG@10 score = 0.6763
AP score      = 0.5183

PERBANDINGAN
Metric       TF-IDF       BM25         Diff        
--------------------------------------------------
RBP          0.5980       0.6353       +0.0373     
DCG@10       2.7983       2.9911       +0.1928     
NDCG@10      0.6431       0.6763       +0.0332     
AP           0.4948       0.5183       +0.0235     


{'tfidf': {'RBP': 0.5979871614743182,
  'DCG@10': 2.7982982589551906,
  'NDCG@10': 0.643064396657458,
  'AP': 0.49483346334616735},
 'bm25': {'RBP': 0.6352773713319504,
  'DCG@10': 2.991060209459272,
  'NDCG@10': 0.6762609744402871,
  'AP': 0.5182941561857902}}

---
## 6. Ringkasan

Notebook ini telah mendemonstrasikan:

| No | Fitur | Deskripsi |
|----|-------|----------|
| 1 | Kompresi OptPForDelta | Algoritma bit-level untuk kompresi index |
| 2 | BM25 Scoring | Scoring dengan length normalization |
| 3 | Metrik DCG, NDCG, AP | Tiga metrik evaluasi tambahan |
| 4 | WAND Top-K | Optimasi retrieval dengan pruning |

Semua fitur tugas utama (100 point) telah diimplementasikan!